In [2]:
import pandas as pd

# Load the Superstore Sales dataset
df = pd.read_csv('train.csv', encoding='latin1')

print(df.shape)
print(df.columns.tolist())
df.head()

(9800, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


Check for missing values

In [3]:
# Check for missing values before any transformation
print("Missing values:")
print(df.isnull().sum())

Missing values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64


In [4]:
# Fill missing Postal Code values with 0 since we won't use this column
# in our dashboard analysis
df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

print("Missing values after fix:")
print(df.isnull().sum())

Missing values after fix:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
dtype: int64


Fix date columns

In [5]:
# Convert Order Date and Ship Date from text to actual datetime format
# Without this Python reads dates as plain text and can't do time analysis
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

print(df['Order Date'].dtype)
print(df[['Order Date', 'Ship Date']].head())

datetime64[ns]
  Order Date  Ship Date
0 2017-11-08 2017-11-11
1 2017-11-08 2017-11-11
2 2017-06-12 2017-06-16
3 2016-10-11 2016-10-18
4 2016-10-11 2016-10-18


Extract time features

In [6]:
# Extract Year, Month, Quarter from Order Date
# These extra columns make time based analysis much easier in Power BI
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%B')
df['Quarter'] = df['Order Date'].dt.quarter

print(df[['Order Date', 'Year', 'Month', 'Month Name', 'Quarter']].head())

  Order Date  Year  Month Month Name  Quarter
0 2017-11-08  2017     11   November        4
1 2017-11-08  2017     11   November        4
2 2017-06-12  2017      6       June        2
3 2016-10-11  2016     10    October        4
4 2016-10-11  2016     10    October        4


Calculate shipping days

In [7]:
# Calculate how many days each order took to ship
# This is a key business metric for measuring delivery performance
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

print(f"Average days to ship: {df['Days to Ship'].mean():.1f} days")
print(df[['Order Date', 'Ship Date', 'Days to Ship']].head())

Average days to ship: 4.0 days
  Order Date  Ship Date  Days to Ship
0 2017-11-08 2017-11-11             3
1 2017-11-08 2017-11-11             3
2 2017-06-12 2017-06-16             4
3 2016-10-11 2016-10-18             7
4 2016-10-11 2016-10-18             7


 Categorize sales

In [8]:
# Categorize orders into size buckets based on sales value
# Makes it easier to analyze order patterns in Power BI
df['Sales Category'] = pd.cut(df['Sales'], 
                               bins=[0, 100, 500, 1000, 10000],
                               labels=['Small', 'Medium', 'Large', 'Very Large'])

print(df['Sales Category'].value_counts())

Sales Category
Small         6104
Medium        2550
Large          684
Very Large     457
Name: count, dtype: int64


Save cleaned dataset

In [9]:
# Save the cleaned and enriched dataset as CSV for Power BI
df.to_csv('superstore_cleaned.csv', index=False)

print("File saved successfully!")
print(f"Final dataset shape: {df.shape}")
print(f"Total columns: {df.columns.tolist()}")

File saved successfully!
Final dataset shape: (9800, 24)
Total columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Year', 'Month', 'Month Name', 'Quarter', 'Days to Ship', 'Sales Category']
